# AI Agent Operations & Performance Analysis

## Project Overview
This notebook analyzes 50,000 AI-agent interaction records to understand operational performance, automation, customer satisfaction, cost, business value, escalation, SLA compliance, and model/department behavior.

**Dataset:** `AI_Agent_Operations_50000_Rows(1).xlsx`

The analysis is descriptive and exploratory and is intended to support an AI-agent operations dashboard/project report.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

FILE = "AI_Agent_Operations_50000_Rows(1).xlsx"
raw = pd.read_excel(FILE, sheet_name="Raw Data")
raw["Date"] = pd.to_datetime(raw["Date"])
raw.head()


## 1. Dataset Overview

In [ ]:
print("Shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())
print("\nMissing values:")
print(raw.isna().sum().sort_values(ascending=False).head(10))
print("\nDuplicate rows:", raw.duplicated().sum())
raw.describe(include="all").T


## 2. Overall KPI Summary

In [ ]:
kpis = {
    "Total Interactions": len(raw),
    "Unique Agents": raw["Agent_ID"].nunique(),
    "Unique Customers": raw["Customer_ID"].nunique(),
    "Departments": raw["Department"].nunique(),
    "Average CSAT": raw["CSAT"].mean(),
    "Average Automation Rate": raw["Automation_Rate"].mean(),
    "Total AI Cost (USD)": raw["AI_Cost_USD"].sum(),
    "Total Business Value (USD)": raw["Business_Value_USD"].sum(),
    "Average Resolution Time (min)": raw["Resolution_Time_Min"].mean(),
    "Escalation Rate": (raw["Escalated"]=="Yes").mean(),
    "Human Review Rate": (raw["Human_Review"]=="Yes").mean(),
    "SLA Breach Rate": (raw["SLA_Breached"]=="Yes").mean(),
}
pd.Series(kpis).to_frame("Value")


## 3. Department Performance

In [ ]:
dept = raw.groupby("Department").agg(
    Interactions=("Interaction_ID","count"),
    Avg_Response_Sec=("Response_Time_Sec","mean"),
    Avg_Resolution_Min=("Resolution_Time_Min","mean"),
    Avg_CSAT=("CSAT","mean"),
    Avg_Automation_Rate=("Automation_Rate","mean"),
    Total_AI_Cost=("AI_Cost_USD","sum"),
    Total_Business_Value=("Business_Value_USD","sum"),
    Escalation_Rate=("Escalated", lambda s: (s=="Yes").mean()),
    Human_Review_Rate=("Human_Review", lambda s: (s=="Yes").mean())
).sort_values("Interactions",ascending=False)
dept


In [ ]:
dept["Interactions"].sort_values().plot(kind="barh", title="Interactions by Department")
plt.xlabel("Interactions"); plt.tight_layout(); plt.show()


## 4. AI Model Performance

In [ ]:
model = raw.groupby("AI_Model").agg(
    Interactions=("Interaction_ID","count"),
    Avg_Success=("Success_Score","mean"),
    Avg_Quality=("Quality_Score","mean"),
    Avg_CSAT=("CSAT","mean"),
    Avg_Automation=("Automation_Rate","mean"),
    Total_Cost=("AI_Cost_USD","sum"),
    Business_Value=("Business_Value_USD","sum"),
    Avg_Response=("Response_Time_Sec","mean")
).sort_values("Avg_Success",ascending=False)
model


In [ ]:
model["Avg_Success"].sort_values().plot(kind="barh", title="Average Success Score by AI Model")
plt.xlabel("Average Success Score"); plt.tight_layout(); plt.show()


## 5. Channel and Regional Analysis

In [ ]:
channel = raw.groupby("Channel").agg(
    Interactions=("Interaction_ID","count"),
    Avg_CSAT=("CSAT","mean"),
    Avg_Automation=("Automation_Rate","mean"),
    Business_Value=("Business_Value_USD","sum")
).sort_values("Interactions",ascending=False)

region = raw.groupby("Region").agg(
    Interactions=("Interaction_ID","count"),
    Avg_CSAT=("CSAT","mean"),
    Avg_Automation=("Automation_Rate","mean"),
    Business_Value=("Business_Value_USD","sum")
).sort_values("Interactions",ascending=False)

display(channel)
display(region)


## 6. Request Type and Priority

In [ ]:
request = raw.groupby("Request_Type").agg(
    Interactions=("Interaction_ID","count"),
    Avg_Success=("Success_Score","mean"),
    Avg_CSAT=("CSAT","mean"),
    Avg_Resolution=("Resolution_Time_Min","mean"),
    Escalation_Rate=("Escalated",lambda s:(s=="Yes").mean())
).sort_values("Interactions",ascending=False)

priority = raw.groupby("Priority").agg(
    Interactions=("Interaction_ID","count"),
    Avg_Success=("Success_Score","mean"),
    Avg_Resolution=("Resolution_Time_Min","mean"),
    Escalation_Rate=("Escalated",lambda s:(s=="Yes").mean())
)

display(request)
display(priority)


## 7. Outcomes, Failures and SLA

In [ ]:
outcomes = raw["Outcome"].value_counts()
failures = raw.loc[raw["Failure_Reason"].notna(),"Failure_Reason"].value_counts()
sla = pd.crosstab(raw["SLA_Breached"], columns="Count")
display(outcomes)
display(failures)
display(sla)


## 8. Daily Operational Trend

In [ ]:
daily = raw.groupby("Date").agg(
    Interactions=("Interaction_ID","count"),
    Avg_CSAT=("CSAT","mean"),
    Business_Value=("Business_Value_USD","sum")
)
daily["Interactions"].plot(title="Daily Interaction Volume", figsize=(10,4))
plt.ylabel("Interactions"); plt.tight_layout(); plt.show()


## 9. Relationship Analysis

In [ ]:
metrics = [
    "Response_Time_Sec","Resolution_Time_Min","Tool_Calls","Total_Tokens",
    "Success_Score","Quality_Score","CSAT","AI_Cost_USD",
    "Automation_Rate","Business_Value_USD","Agent_Efficiency_Score"
]
corr = raw[metrics].corr()
plt.figure(figsize=(12,9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Key Operational Metrics")
plt.tight_layout()
plt.show()


## 10. Key Findings

In [ ]:
print("Highest interaction department:", dept["Interactions"].idxmax())
print("Highest business-value department:", dept["Total_Business_Value"].idxmax())
print("Highest automation department:", dept["Avg_Automation_Rate"].idxmax())
print("Highest CSAT department:", dept["Avg_CSAT"].idxmax())
print("Highest escalation department:", dept["Escalation_Rate"].idxmax())
print("Highest average-success AI model:", model["Avg_Success"].idxmax())
print("Total business value: $", round(raw["Business_Value_USD"].sum(),2))
print("Total AI cost: $", round(raw["AI_Cost_USD"].sum(),2))
